In [ ]:
import numpy as np
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
import matplotlib.pyplot as plt

from emu_renewal.inputs import OUTPUTS_PATH, get_world_shp
from emu_renewal.outputs import get_prop_improve, get_param_vals_by_analysis
from emu_renewal.plotting import MOB_SOURCE_MAP, MOB_COLOURS, ANALYSIS_TYPES

In [ ]:
plt.style.use("default")
set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "44255911"
countries = ls(job_path)
world = get_world_shp()

In [ ]:
category_colors = {
    "no_mob": "red",
    "g_mob": "green",
    "a_mob": "red",
    "fb_mob": "blue",
}

In [ ]:
param = "dispersion_proc"
posts = {}
for c in countries[:10]:
    posts[c] = get_param_vals_by_analysis(param, job_path / c)
    best_mob = posts[c].mean().idxmin()
    world.loc[world["ISO_A3"] == c, "best_mob"] = best_mob
    world["colour"] = world["best_mob"].map(category_colors)
world.loc[world["colour"].isna(), "colour"] = "grey"

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 7), constrained_layout=True)
axes = axs.ravel()
for a, mob_type in enumerate(ANALYSIS_TYPES[1:]):
    ax = axes[a + 1]
    ax.set_title(MOB_SOURCE_MAP[mob_type])

    prop_improve = {}
    for c in countries[:10]:
        c_posts = posts[c]
        if mob_type in c_posts:
            c_posts[mob_type] = np.random.permutation(c_posts[mob_type].values)
            prop_improve[c] = pd.Series(c_posts["no_mob"] > c_posts[mob_type]).astype(int).mean()
    world[f"{mob_type}_improve"] = world["ISO_A3"].map(prop_improve)
    world.plot(
        column=f"{mob_type}_improve",
        ax=ax,
        cmap=MOB_COLOURS[mob_type].capitalize() + "s",
        legend=True,
        missing_kwds={"color": "lightgrey"},
        vmin=0.0,
        vmax=1.0,
    )

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    world.boundary.plot(ax=ax, color="black", linewidth=0.2)

ax = axes[0]
ax.set_title("Best mobility type")
world.plot(column="colour", ax=ax)